# Equivariant GNN Training Demo

Train an `ExceptionalEGNN` on a synthetic task: predict the total Killing norm of node features.
Uses G2 (dim=14) for fast iteration.

In [ ]:
!pip install -q dhl-mm torch scipy

## Synthetic Dataset

Each graph has random G2-valued node features and random edges.
Target: sum of Killing self-norms K(x_i, x_i) — an adjoint-invariant quantity.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from equivariant import ExceptionalEGNN, SparseKillingForm

def make_graph(n_nodes=10, algebra_name='G2'):
    dim = {'G2': 14, 'F4': 52, 'E6': 78, 'E7': 133, 'E8': 248}[algebra_name]
    x = torch.randn(n_nodes, dim) * 0.1
    edges = []
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if torch.rand(1) < 0.3:
                edges.extend([(i,j), (j,i)])
    if not edges:
        edges = [(0,1), (1,0)]
    edge_index = torch.tensor(edges, dtype=torch.long).T
    killing = SparseKillingForm.from_algebra(algebra_name)
    target = sum(killing(x[i], x[i]) for i in range(n_nodes))
    return x, edge_index, target.unsqueeze(0)

dataset = [make_graph(n_nodes=8, algebra_name='G2') for _ in range(200)]
print(f'Dataset: {len(dataset)} graphs')
print(f'Sample: {dataset[0][0].shape} nodes, {dataset[0][1].shape[1]} edges, target={dataset[0][2].item():.4f}')

## Train Equivariant Model

3 equivariant layers with Schur's-lemma-constrained linear maps and bracket-based nonlinearities.

In [ ]:
model = ExceptionalEGNN(
    in_dim=14, hidden_dim=32, out_dim=1,
    n_layers=3, algebra_name='G2', equivariant=True
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(100):
    total_loss = 0
    for x, edge_index, target in dataset[:100]:
        pred = model(x, edge_index)
        loss = (pred - target).pow(2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    losses.append(total_loss / 100)
    if epoch % 20 == 0:
        print(f'Epoch {epoch}: loss = {losses[-1]:.4f}')

print(f'\nFinal: {losses[-1]:.4f} (started at {losses[0]:.4f}, {losses[0]/losses[-1]:.1f}x reduction)')

## Evaluate

In [ ]:
model.eval()
test_losses = []
with torch.no_grad():
    for x, edge_index, target in dataset[100:]:
        pred = model(x, edge_index)
        test_losses.append((pred - target).pow(2).item())

print(f'Train MSE: {losses[-1]:.4f}')
print(f'Test MSE:  {np.mean(test_losses):.4f}')

---

`pip install dhl-mm` | [GitHub](https://github.com/grapheneaffiliate/DHL-MM) | [PyPI](https://pypi.org/project/dhl-mm/)